In [44]:
import math
import gurobipy as gurobi
from gurobipy import GRB

### Problem 1: Economic Dispatch

$p_i$: output of plant $i$ (MW)

$$\min \sum_{i} a_i p_i^2 + b_i p_i$$

$$\sum_{i} p_i = 600, \quad 0 \le p_i \le \text{Cap}_i$$

In [45]:
plants = {1: (0.02, 8, 350), 2: (0.04, 12, 200), 3: (0.05, 15, 200), 4: (0.025, 5, 150)}
demand = 600

m = gurobi.Model("economic_dispatch")

p = {i: m.addVar(lb=0, ub=cap, name=f"p_{i}") for i, (a, b, cap) in plants.items()}

m.setObjective(gurobi.quicksum(a * p[i] * p[i] + b * p[i] for i, (a, b, cap) in plants.items()), GRB.MINIMIZE)

m.addConstr(gurobi.quicksum(p[i] for i in plants) == demand, name="demand")

m.optimize()

print(f"\nOptimal total cost: ${m.ObjVal:,.2f} per hour\n")
for i, (a, b, cap) in plants.items():
    print(f"  Plant {i}: {p[i].X:.1f} MW (capacity {cap} MW)")

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26300.2))

CPU model: AMD Ryzen 3 3200U with Radeon Vega Mobile Gfx, instruction set [SSE2|AVX|AVX2]
Thread count: 2 physical cores, 4 logical processors, using up to 4 threads

Optimize a model with 1 rows, 4 columns and 4 nonzeros (Min)
Model fingerprint: 0x0364238c
Model has 4 linear objective coefficients
Model has 4 quadratic objective terms
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [5e+00, 2e+01]
  QObjective range [4e-02, 1e-01]
  Bounds range     [2e+02, 4e+02]
  RHS range        [6e+02, 6e+02]

Presolve time: 0.01s
Presolved: 1 rows, 4 columns, 4 nonzeros
Presolved model has 4 quadratic objective terms
Ordering time: 0.00s

Barrier statistics:
 AA' NZ     : 0.000e+00
 Factor NZ  : 1.000e+00
 Factor Ops : 1.000e+00 (less than 1 second per iteration)
 Threads    : 1

                  Objective                Residual
Iter       Primal          Dual         Primal    Dual 

In [46]:
def plant_cost(i, q):
    a, b, cap = plants[i]
    return a * q ** 2 + b * q


equal_cost = sum(plant_cost(i, 150) for i in plants)
print(f"Cost with 150 MW at every plant: ${equal_cost:,.2f} per hour")
print(f"Savings from the optimal plan: ${equal_cost - m.ObjVal:,.2f} per hour\n")

print("Marginal cost 2a*p + b at the optimal plan:")
for i, (a, b, cap) in plants.items():
    print(f"  Plant {i}: {2 * a * p[i].X + b:.2f} $/MWh")

Cost with 150 MW at every plant: $9,037.50 per hour
Savings from the optimal plan: $1,050.00 per hour

Marginal cost 2a*p + b at the optimal plan:
  Plant 1: 20.00 $/MWh
  Plant 2: 20.00 $/MWh
  Plant 3: 20.00 $/MWh
  Plant 4: 12.50 $/MWh


### Problem 3: Gradient Descent for Fitting a Line

$$\min f(a, b) = \sum_{i} (y_i - a - b x_i)^2$$

$$\nabla f(a, b) = (10a + 20b - 46,\ 20a + 60b - 128)$$

In [47]:
xs = [0, 1, 2, 3, 4]
ys = [1, 3, 4, 7, 8]


def f(a, b):
    return sum((y - a - b * x) ** 2 for x, y in zip(xs, ys))


def grad(a, b):
    da = sum(-2 * (y - a - b * x) for x, y in zip(xs, ys))
    db = sum(-2 * x * (y - a - b * x) for x, y in zip(xs, ys))
    return da, db


def gradient_descent(alpha, max_iter=1000, tol=0.01, show=0):
    a, b = 0.0, 0.0
    ga, gb = grad(a, b)
    k = 0
    while math.hypot(ga, gb) >= tol and k < max_iter:
        a, b = a - alpha * ga, b - alpha * gb
        ga, gb = grad(a, b)
        k += 1
        if k <= show:
            print(f"  Iteration {k}: (a, b) = ({a:.4f}, {b:.4f}), f = {f(a, b):.4f}, ||grad f|| = {math.hypot(ga, gb):.4f}")
    return a, b, k, math.hypot(ga, gb)


print("Learning rate 0.01:")
a, b, k, g = gradient_descent(0.01, show=2)
print(f"\nIterations: {k}")
print(f"a = {a:.4f}, b = {b:.4f}, f = {f(a, b):.4f}, ||grad f|| = {g:.4f}")

Learning rate 0.01:
  Iteration 1: (a, b) = (0.4600, 1.2800), f = 15.9860, ||grad f|| = 44.8736
  Iteration 2: (a, b) = (0.6180, 1.7000), f = 2.5936, ||grad f|| = 14.8298

Iterations: 154
a = 0.9969, b = 1.8011, f = 0.8000, ||grad f|| = 0.0098


In [48]:
m = gurobi.Model("line_fit")

a_var = m.addVar(lb=-GRB.INFINITY, name="a")
b_var = m.addVar(lb=-GRB.INFINITY, name="b")

m.setObjective(gurobi.quicksum((y - a_var - b_var * x) * (y - a_var - b_var * x) for x, y in zip(xs, ys)), GRB.MINIMIZE)

m.optimize()

print(f"\nGurobi: a = {a_var.X:.4f}, b = {b_var.X:.4f}, f = {m.ObjVal:.4f}")

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26300.2))

CPU model: AMD Ryzen 3 3200U with Radeon Vega Mobile Gfx, instruction set [SSE2|AVX|AVX2]
Thread count: 2 physical cores, 4 logical processors, using up to 4 threads

Optimize a model with 0 rows, 2 columns and 0 nonzeros (Min)
Model fingerprint: 0x0d1742f4
Model has 2 linear objective coefficients and an objective constant of 139
Model has 3 quadratic objective terms
Coefficient statistics:
  Matrix range     [0e+00, 0e+00]
  Objective range  [5e+01, 1e+02]
  QObjective range [1e+01, 6e+01]
  Bounds range     [0e+00, 0e+00]
  RHS range        [0e+00, 0e+00]

Presolve time: 0.02s
Presolved: 0 rows, 2 columns, 0 nonzeros
Presolved model has 3 quadratic objective terms
Ordering time: 0.00s

Barrier statistics:
 Free vars  : 3
 AA' NZ     : 0.000e+00
 Factor NZ  : 1.000e+00
 Factor Ops : 1.000e+00 (less than 1 second per iteration)
 Threads    : 1

                  Objective                Residual
Iter  

(d) Learning rates 0.005, 0.01, 0.02, and 0.035, with a cap of 1000 iterations.

In [49]:
for alpha in [0.005, 0.01, 0.02, 0.035]:
    a, b, k, g = gradient_descent(alpha)
    status = "converged" if g < 0.01 else "stopped at cap"
    print(f"alpha = {alpha}: {k} iterations ({status}), a = {a:.4g}, b = {b:.4g}, f = {f(a, b):.4g}, ||grad f|| = {g:.4g}")

alpha = 0.005: 309 iterations (converged), a = 0.9968, b = 1.801, f = 0.8, ||grad f|| = 0.009969
alpha = 0.01: 154 iterations (converged), a = 0.9969, b = 1.801, f = 0.8, ||grad f|| = 0.009768
alpha = 0.02: 76 iterations (converged), a = 0.9969, b = 1.801, f = 0.8, ||grad f|| = 0.009658
alpha = 0.035: 1000 iterations (stopped at cap), a = -5.321e+128, b = -1.517e+129, f = 8.66e+259, ||grad f|| = 1.077e+131
